# 03 · Evaluate on the held-out event

Scores three ways of producing the after image of the **test event** (never seen in training):

| Method | What it is |
|---|---|
| **pix2pix** (ours) | trained on Kerala before/after pairs with slope, rain, soil, severity |
| **Identity** baseline | "nothing changes": after = before. pix2pix must beat this. |
| **Stable Diffusion img2img** | large pretrained diffusion model, no training on this task, text prompt only |

Metrics: SSIM, PSNR, FID (all on RGB), and **severity consistency** for pix2pix (does the
measured damage follow the requested severity?). Results go to `outputs/metrics.json`, which
is the only source of numbers for the report. Runtime: **GPU (T4)**.

In [ ]:
from google.colab import drive

REPO_URL = "https://github.com/muhammedshahidsds25-collab/disaster-gen.git"

drive.mount("/content/drive")
!git clone -q {REPO_URL} /content/disaster-gen || git -C /content/disaster-gen pull -q
%cd /content/disaster-gen
!pip install -q -e ".[diffusion]"

In [ ]:
from pathlib import Path

import pandas as pd
import torch

from disastergen.config import load_config
from disastergen.data.dataset import (
    ChannelStats,
    TileDataset,
    split_by_event,
    to_reflectance,
    unnormalise_before,
)
from disastergen.diffusion_baseline import load_pipeline, render_prompts, sd_img2img
from disastergen.evaluate import (
    SEVERITY_CHANNEL,
    evaluate_identity,
    evaluate_model,
    evaluate_rgb,
    plot_severity_sweep,
    plot_triplets,
    rgb_uint8,
    save_metrics,
)
from disastergen.models import UNetGenerator

cfg = load_config("configs/default.yaml")
DRIVE_ROOT = Path("/content/drive/MyDrive") / cfg.data.drive_folder
CKPT_DIR = DRIVE_ROOT / "checkpoints"
OUT_DIR = DRIVE_ROOT / "outputs"
(OUT_DIR / "figures").mkdir(parents=True, exist_ok=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

stats = ChannelStats.load(CKPT_DIR / "stats.json")
split = split_by_event(
    sorted((DRIVE_ROOT / "tiles").glob("*.npz")), {cfg.data.test_event}, cfg.train.val_frac
)
test_ds = TileDataset(split["test"], stats, augment=False)
test_event = next(e for e in cfg.events if e.name == cfg.data.test_event)

g = UNetGenerator(cfg.model.in_channels, cfg.model.out_channels, cfg.model.ngf, cfg.model.num_downs)
g.load_state_dict(torch.load(CKPT_DIR / "best_g.pt", map_location=device, weights_only=True))
g = g.to(device).eval()
print(f"Test event: {test_event.name} ({test_event.hazard}), {len(test_ds)} tiles")

## 1. pix2pix vs. identity baseline

In [ ]:
results = {
    "test_event": test_event.name,
    "pix2pix": evaluate_model(g, test_ds, cfg, device),
    "identity": evaluate_identity(test_ds, cfg),
}
sweep = results["pix2pix"].pop("severity_sweep")  # raw points: plotted below, not in the table
pd.DataFrame({k: results[k] for k in ("pix2pix", "identity")}).T

## 2. Figures

In [ ]:
@torch.no_grad()
def generate(x: torch.Tensor) -> torch.Tensor:
    return to_reflectance(g(x.to(device)).cpu(), stats)


severe_first = test_ds.severities.argsort()[::-1][:4].tolist()
xs, ys = zip(*(test_ds[i] for i in severe_first), strict=True)
x, y = torch.stack(xs), torch.stack(ys)
before_rgb = rgb_uint8(unnormalise_before(x, stats), cfg.eval.rgb_max)
real_rgb = rgb_uint8(to_reflectance(y, stats), cfg.eval.rgb_max)
plot_triplets(before_rgb, real_rgb, rgb_uint8(generate(x), cfg.eval.rgb_max),
              OUT_DIR / "figures" / "triplets.png")

In [ ]:
frames = []
for sev in cfg.demo.sweep:
    swept = x[:1].clone()
    swept[:, SEVERITY_CHANNEL] = sev
    frames.append(rgb_uint8(generate(swept), cfg.eval.rgb_max)[0])
plot_severity_sweep(before_rgb[0], frames, cfg.demo.sweep, OUT_DIR / "figures" / "sweep.png")

In [ ]:
sweep_df = pd.DataFrame(sweep)
ax = sweep_df.boxplot(column="measured", by="requested", figsize=(6, 4))
ax.set_xlabel("requested severity")
ax.set_ylabel("measured severity of generated image")
ax.figure.suptitle("")
ax.figure.savefig(OUT_DIR / "figures" / "severity_consistency.png", dpi=150)

## 3. Stable Diffusion img2img (zero-shot, prompt engineering)
Same before images, 4 prompt designs from `configs/default.yaml`. Takes ~20-30 min on a T4.

In [ ]:
pipe = load_pipeline(cfg.eval, device)
all_x, all_y = zip(*(test_ds[i] for i in range(len(test_ds))), strict=True)
test_before_rgb = rgb_uint8(unnormalise_before(torch.stack(all_x), stats), cfg.eval.rgb_max)
test_real_rgb = rgb_uint8(to_reflectance(torch.stack(all_y), stats), cfg.eval.rgb_max)

results["stable_diffusion"] = {}
sd_examples = {}
for prompt in render_prompts(cfg.eval.prompts, test_event.hazard):
    sd_rgb = sd_img2img(pipe, test_before_rgb, prompt, cfg.eval, seed=cfg.train.seed)
    results["stable_diffusion"][prompt] = evaluate_rgb(sd_rgb, test_real_rgb)
    sd_examples[prompt] = sd_rgb[severe_first]
pd.DataFrame(results["stable_diffusion"]).T

In [ ]:
best_prompt = max(results["stable_diffusion"], key=lambda p: results["stable_diffusion"][p]["ssim"])
results["stable_diffusion_best_prompt"] = best_prompt
plot_triplets(before_rgb, real_rgb, sd_examples[best_prompt],
              OUT_DIR / "figures" / "triplets_stable_diffusion.png")
best_prompt

## 4. Results table → `outputs/metrics.json`

In [ ]:
save_metrics(results, OUT_DIR / "metrics.json")
table = {"pix2pix (ours)": results["pix2pix"], "identity baseline": results["identity"],
         f"Stable Diffusion ({best_prompt!r})": results["stable_diffusion"][best_prompt]}
pd.DataFrame(table).T[["n_tiles", "ssim", "psnr", "fid"]].round(3)

Copy `outputs/metrics.json` and `outputs/figures/` from Drive into the repo's `outputs/`
folder, and save this executed notebook into `notebooks/`, then commit.